In [1]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

2026-04-02 09:06:37.126445: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


3.12.6 (main, Sep 30 2024, 02:19:13) [GCC 9.4.0]
pandas 2.3.3
polars 1.39.0
numpy 2.1.3
matplotlib.pyplot
seaborn 0.13.2
scipy 1.15.2
sklearn 1.5.2
tensorflow 2.20.0
xgboost 3.2.0
lightgbm 4.6.0
catboost 1.2.8
mllabs 0.6.4


In [2]:
from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

In [3]:
from sklearn.linear_model import LinearRegression
X_reg = ['PhoneService', 'DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 
        'StreamingTV_Y', 'TechSupport_Y', 'No_Internet', 'DSL_Y', 'MultipleLines_Y', 'SeniorCitizen']

reg_lr = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: LinearRegression(fit_intercept = True).fit(x[X_reg], x['MonthlyCharges'])
)
reg_lgb = df_org[X_reg + ['MonthlyCharges']].drop_nulls().pipe(
    lambda x: lgb.LGBMRegressor(verbose=-1).fit(x[X_reg].to_pandas(), x['MonthlyCharges'].to_pandas())
)

In [4]:
from sklearn.model_selection import cross_val_score
cross_val_score(
    LinearRegression(fit_intercept = True), df_org[X_reg], df_org['MonthlyCharges'], scoring = 'r2', cv = 5
)

array([0.99884924, 0.99888459, 0.998819  , 0.99880339, 0.99881452])

In [5]:
cross_val_score(
   lgb.LGBMRegressor(verbose=-1), df_org[X_reg].to_pandas(), df_org['MonthlyCharges'].to_pandas(), scoring = 'r2', cv = 5
)

array([0.99875274, 0.99882932, 0.99875319, 0.99874725, 0.99871514])

In [6]:
df_train = df_train.with_columns(
    MonthlyCharges_exp_lr = reg_lr.predict(df_train[X_reg]),
    MonthlyCharges_exp_lgb = reg_lgb.predict(df_train[X_reg])
).with_columns(
    TotalCharges_exp_lr = pl.col('MonthlyCharges_exp_lr') * pl.col('tenure'),
    tenure_exp_lr = pl.col('TotalCharges') / (pl.col('MonthlyCharges_exp_lr') + 1),
    TotalCharges_exp_lgb = pl.col('MonthlyCharges_exp_lgb') * pl.col('tenure'),
    tenure_exp_lgb = pl.col('TotalCharges') / pl.col('MonthlyCharges_exp_lgb')
)
X_exp = ['MonthlyCharges_exp_lr', 'MonthlyCharges_exp_lgb', 'TotalCharges_exp_lr', 'TotalCharges_exp_lgb', 'tenure_exp_lr', 'tenure_exp_lgb']
X_exp_lr = ['MonthlyCharges_exp_lr', 'TotalCharges_exp_lr', 'tenure_exp_lr']
X_exp_lgb = ['MonthlyCharges_exp_lgb', 'TotalCharges_exp_lgb', 'tenure_exp_lgb']

In [9]:
from mllabs import ProgressSessionLogger, TqdmProgressSession
if os.path.exists('exp/ss1_2'):
    e_ss1_2 = Experimenter.load('exp/ss1_2', df_train)
    if e_ss1_2.status == 'closed':
        e_ss1_2.reopen_exp()
else:
    e_ss1_2 = Experimenter.create(
        df_train, 'exp/ss1_2', title='The experimentation using Shuffle Split',
        sp=StratifiedShuffleSplit(n_splits=1, random_state=1),
        splitter_params={'y': target}, logger=ProgressSessionLogger(level=['info', 'progress'], session_cls=TqdmProgressSession)
    )

e_ss1_2.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_ss1_2.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)
e_ss1_2.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)
e_ss1_2.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01
    }
)

e_ss1_2.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)

e_ss1_2.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'learning_rate': 0.05, 'verbose': 0, 'random_state': 1, 'max_depth': 5, 'colsample_bylevel': 0.75
    }
)

## Neural network
e_ss1_2.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 0, 'validation_fraction': 0})

Markdown(
    e_ss1_2.desc_spec()
)

📁 Created directory: exp/ss1_2


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

## The experimentation using Shuffle Split

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedShuffleSplit(n_splits=1, random_state=1)` |
| **Inner Splitter (sp_v)** | None |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 1 |
| **Inner Folds** | 1 |

In [10]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_ss1_2.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_ss1_2.set_node(
    'kbin', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', None)]}, params = {'n_bins': 10, 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_ss1_2.set_node(
    'kbin_1', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])]},
    params = {'n_bins': [4000, 200], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_ss1_2.set_node(
    'kbin_2', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])]},
    params = {'n_bins': [500, 100], 'strategy': 'uniform', 'encode': 'ordinal',  'random_state': 123}
)
e_ss1_2.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_ss1_2.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_ss1_2.set_node(
    'tgt_num', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])], **y_edges}, params = {'target_type': 'binary'}
)
e_ss1_2.set_node(
    'freq_num', grp='pre', processor=FrequencyEncoder, method = 'transform', edges = {'X': [('si', ['.*TotalCharges$', '.*MonthlyCharges$'])], **y_edges}, params = {'normalize': True}
)
e_ss1_2.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)
e_ss1_2.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin', None), ('kbin_1', None), ('kbin_2', None)]}
)
e_ss1_2.set_node(
    'b2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [(None, X_bin), (None, X_bin2)]}, params={'to':'str'}
)
e_ss1_2.set_node(
    'b2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('b2s', None)]}
)

e_ss1_2.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)

e_ss1_2.set_node(
    'rb_std', grp='pre', processor=RobustScaler, edges = {'X': [('rb', None)]}, params = {}
)
e_ss1_2.set_node(
    'expr4', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'Monthly_per_Total': pl.col('si__MonthlyCharges') / pl.col('si__TotalCharges'),
        'Monthly_Avg_ratio': pl.col('si__MonthlyCharges') / (pl.col('si__TotalCharges') / pl.col('si__tenure')),
        'tenure_sq': pl.col('si__tenure') ** 2,
    }, 'with_columns': False}
)
service = ['DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 'StreamingTV_Y', 'TechSupport_Y']
e_ss1_2.set_node(
    'expr5', grp='pre', processor = ExprProcessor, edges = {'X': [(None, service)]},
    params={'dict_expr': {
        'Service_cnt':  pl.sum_horizontal(service),
    }, 'with_columns': False}
)
e_ss1_2.set_node(
    'expr4_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr4', None)]}
)
e_ss1_2.set_node(
    'exp_std', grp='pre', processor = StandardScaler, edges = {'X': [(None, X_exp)]}
)
e_ss1_2.build()

Building 18 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 18 node(s)


In [11]:
from mllabs.processor import CatPairCombiner
from itertools import combinations

TOP_CATS_FOR_NGRAM = ['Contract', 'DSL_Y', 'PaymentMethod', 'OnlineSecurity']

bigram_cols = [(i1, i2) for i1, i2 in combinations(TOP_CATS_FOR_NGRAM, 2)]
trigram_cols = [(i1, i2, i3) for i1, i2, i3 in combinations(TOP_CATS_FOR_NGRAM, 3)]

e_ss1_2.set_node(
    'bigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM)]}, params={'pairs': bigram_cols}
)
e_ss1_2.set_node(
    'trigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM[:4])]}, params={'pairs': trigram_cols}
)
e_ss1_2.set_node(
    'com3', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, ['Contract', 'InternetService', 'PaymentMethod'])]}, 
    params={'pairs': [('Contract', 'InternetService', 'PaymentMethod')]}
)
e_ss1_2.set_node(
    'tgt_bigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('bigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_ss1_2.set_node(
    'tgt_trigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('trigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_ss1_2.set_node(
    'coov_bigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('bigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_ss1_2.set_node(
    'coov_trigram', grp='pre', processor = CatOOVFilter, edges = {'X': [('trigram', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_ss1_2.set_node(
    'exprd', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_3': ((pl.col('si__TotalCharges') * 1e-3).cast(pl.Int8) % 10).cast(pl.String).cast(pl.Categorical)
    }, 'with_columns': False}
)
e_ss1_2.set_node(
    'n2c_2', grp='pre', processor = ExprProcessor, edges = {'X': [('si', ['.*Total.*', '.*Month.*'])]},
    params={'dict_expr': {
        'TotalCharges': pl.col('si__TotalCharges').round(0).cast(pl.String).cast(pl.Categorical),
        'MonthlyCharges': pl.col('si__MonthlyCharges').round(0).cast(pl.String).cast(pl.Categorical)
    }, 'with_columns': False}
)
e_ss1_2.set_node(
    'coov2', grp='pre', processor = CatOOVFilter, edges = {'X': [('n2c_2', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)
e_ss1_2.build()

Building 10 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Build complete: 10 node(s)


In [12]:
i = 9
e_ss1_2.set_node(
    'lgb_{}'.format(i), grp = 'lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num + X_exp), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params={'num_leaves': 7, 'n_estimators': 3000, 'learning_rate': 0.05, 'colsample_bytree':0.5}
)
e_ss1_2.exp()
e_ss1_2.get_collector('AUC').get_metrics_agg('lgb_{}'.format(i))[0].sort_values('test', ascending = False)

Experimenting 1 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Exp complete: 1 node(s)


,test,train
lgb_9,0.917871,0.923264


In [13]:
e_ss1_2.show_error_nodes(traceback=True)

No error nodes found


In [14]:
e_ss1_2.set_node(
    'nn_0', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('expr4_std', None), ('coov', None), ('exprd', None), ('coov_bigram', None), ('coov_trigram', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 4, 'learning_rate': 0.0001}
)
e_ss1_2.exp()
e_ss1_2.get_collector('AUC').get_metrics_agg('nn_0'.format(i))[0].sort_values('test', ascending = False)

Experimenting 1 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

I0000 00:00:1775120846.308087    9941 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4600 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5


Exp complete: 1 node(s)


,test,train
nn_0,0.917092,0.925367


In [15]:
e_ss1_2.set_node(
    'nn_1', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('exp_std', None), ('expr4_std', None), ('coov', None), ('exprd', None), ('coov_bigram', None), ('coov_trigram', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [256, 128], 'dropout': 0.5}, 'cat_cols': ColSelector(col_type = 'category'), 'epochs': 4, 'learning_rate': 0.0001}
)
e_ss1_2.exp()
e_ss1_2.get_collector('AUC').get_metrics_agg('nn_1'.format(i))[0].sort_values('test', ascending = False)

Experimenting 1 node(s)


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Exp complete: 1 node(s)


,test,train
nn_1,0.917263,0.925347


In [16]:
e_ss1_2.finalize(None)

Finalize 'lgb_9'
Finalize 'nn_0'
Finalize 'nn_1'
